# Partie 2 — Nettoyage Spark & Analyse NLP (MapReduce)
**Entrée** : `immobilier_fusionne.csv`  
**Sortie** : `immobilier_clean.csv`

## 1. Installation & Imports

In [1]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pyspark','--quiet'])
print('OK')

OK



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, shutil, re, subprocess, time
import pandas as pd

subprocess.run('pkill -f SparkSubmit',   shell=True, capture_output=True)
subprocess.run('pkill -f pyspark',       shell=True, capture_output=True)
subprocess.run('pkill -f GatewayServer', shell=True, capture_output=True)
time.sleep(2)

try:
    from pyspark import SparkContext
    if SparkContext._active_spark_context:
        SparkContext._active_spark_context.stop()
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
except Exception:
    pass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from pyspark.ml.feature import Imputer

spark = SparkSession.builder \
    .appName('Immobilier') \
    .config('spark.driver.memory', '2g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.driver.maxResultSize', '1g') \
    .config('spark.memory.offHeap.enabled', 'true') \
    .config('spark.memory.offHeap.size', '512m') \
    .master('local[2]') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
DATA_DIR = os.getcwd()
print(f'Spark {spark.version} — OK')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/12 18:08:57 WARN Utils: Your hostname, codespaces-8db3e8, resolves to a loopback address: 127.0.0.1; using 10.0.0.247 instead (on interface eth0)
26/04/12 18:08:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 18:08:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 — OK


## 2. Chargement

In [3]:
df = spark.read.csv(
    os.path.join(DATA_DIR, 'immobilier_fusionne.csv'),
    header=True, inferSchema=True
)
print(f'Lignes chargees : {df.count()}')
df.printSchema()

Lignes chargees : 2494
root
 |-- type_transaction: string (nullable = true)
 |-- type_bien: string (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- nb_pieces: double (nullable = true)
 |-- nb_chambres: double (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- source: string (nullable = true)



## 3. Suppression des valeurs manquantes & imputation

In [4]:
# Suppression nulls + filtre transactions
avant = df.count()
df = df.dropna(subset=['prix', 'surface'])
df = df.filter(F.col('type_transaction').isin('vente', 'location'))

# Imputer necessite DoubleType
df = df.withColumn('nb_pieces', F.col('nb_pieces').cast(DoubleType()))
imputer = Imputer(inputCols=['nb_pieces'], outputCols=['nb_pieces'], strategy='median')
df = imputer.fit(df).transform(df)

df = df.withColumn('ville',       F.initcap(F.trim(F.col('ville')))) \
       .withColumn('code_postal', F.trim(F.col('code_postal')))
print(f'Apres nulls  : {df.count()} lignes (supprime {avant - df.count()})') 

Apres nulls  : 2416 lignes (supprime 78)


## 4. Suppression des doublons

In [5]:
# Deduplication
avant = df.count()
window_dedup = Window.partitionBy(
    'type_transaction','type_bien','prix','surface','ville'
).orderBy(F.col('description').isNull().asc())

df = df.withColumn('_rank', F.row_number().over(window_dedup)) \
       .filter(F.col('_rank') == 1) \
       .drop('_rank')
print(f'Apres dedup  : {df.count()} lignes (supprime {avant - df.count()})') 

Apres dedup  : 1844 lignes (supprime 572)


## 5. Filtrage des outliers (IQR × 3)

In [6]:
# Filtrage outliers IQR x3
def filter_iqr(df, col_name, group_col='type_transaction', factor=3.0):
    quantiles = df.groupBy(group_col).agg(
        F.expr(f'percentile_approx({col_name}, 0.25)').alias('q1'),
        F.expr(f'percentile_approx({col_name}, 0.75)').alias('q3')
    ).withColumn('iqr',   F.col('q3') - F.col('q1')) \
     .withColumn('lower', F.col('q1') - factor * F.col('iqr')) \
     .withColumn('upper', F.col('q3') + factor * F.col('iqr')) \
     .select(group_col, 'lower', 'upper')
    quantiles.show()
    return df.join(quantiles, on=group_col, how='left') \
             .filter((F.col(col_name) >= F.col('lower')) &
                     (F.col(col_name) <= F.col('upper'))) \
             .drop('lower', 'upper')

avant = df.count()
df.cache()
df.count()

df = filter_iqr(df, 'prix')
df = filter_iqr(df, 'surface')
df = df.filter((F.col('surface') >= 5) & (F.col('surface') <= 1000))
df = df.filter(F.col('prix') > 0)
df.cache()
df.count()
print(f'Apres IQR    : {df.count()} lignes (supprime {avant - df.count()})') 

+----------------+---------+---------+
|type_transaction|    lower|    upper|
+----------------+---------+---------+
|        location|  -1670.0|   3650.0|
|           vente|-674000.0|1433000.0|
+----------------+---------+---------+

+----------------+------+-----+
|type_transaction| lower|upper|
+----------------+------+-----+
|        location|-121.0|229.0|
|           vente|-148.0|321.0|
+----------------+------+-----+

Apres IQR    : 1790 lignes (supprime 54)


## 6. Bilan du dataset nettoyé

In [7]:
# Bilan
print('=' * 50)
print('BILAN — DATASET NETTOYE')
print('=' * 50)
df.groupBy('source', 'type_transaction').count().orderBy('source').show()
df.select('prix', 'surface', 'nb_pieces').describe().show()
n_desc = df.filter(F.col('description').isNotNull()).count()
print(f'Avec description : {n_desc} / {df.count()} ({n_desc/df.count()*100:.1f}%)')

BILAN — DATASET NETTOYE
+--------------------+----------------+-----+
|              source|type_transaction|count|
+--------------------+----------------+-----+
| 2 chambres dans ...|           vente|    1|
| 2 chambres. 3 sq...|        location|    1|
| au calme de 24 m...|        location|    1|
| au sein de la co...|           vente|    1|
|        centre-ville|           vente|    1|
| dans un immeuble...|        location|    1|
| dans une Villa s...|           vente|    1|
|  dans une rue calme|           vente|    1|
| je propose à la ...|           vente|    1|
| libre en juin 20...|        location|    1|
| une cuisine semi...|           vente|    1|
| une salle de bai...|           vente|    1|
| une véritable ca...|           vente|    1|
| à l'étage : déga...|           vente|    1|
| à la limite de l...|           vente|    1|
| à proximité immé...|           vente|    1|
|   entreparticuliers|        location|  197|
|   entreparticuliers|           vente|   87|
|         

## 7. Analyse NLP des descriptions (MapReduce)

- **Map** : tokenisation → `(mot, 1)` par description
- **Reduce** : agrégation → fréquence totale par mot

In [8]:
STOP_WORDS = {
    'le','la','les','de','du','des','un','une','et','en','au','aux','a','l',
    'se','sa','son','sur','sous','par','pour','avec','dans','est','ce','qui',
    'que','il','elle','ils','elles','vous','nous','on','je','tu','y','ou',
    'si','mais','donc','car','ni','ne','pas','plus','d','qu','n','c','j',
    'tres','bien','tout','cette','cet','ses','leur','leurs'
}
stop_bc = spark.sparkContext.broadcast(STOP_WORDS)

def tokenize(text):
    stop = stop_bc.value
    words = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    return [(w, 1) for w in words if len(w) > 2 and w not in stop]

descriptions_rdd = df.filter(F.col('description').isNotNull()) \
                     .select('description') \
                     .rdd.map(lambda r: r['description'])

K = 30
top_k = descriptions_rdd.flatMap(tokenize) \
                         .reduceByKey(lambda a, b: a + b) \
                         .sortBy(lambda x: x[1], ascending=False) \
                         .take(K)

print(f'Top {K} mots (RDD MapReduce) :')
print(f'{"Rang":<6} {"Mot":<20} {"Freq":>6}')
print('-' * 35)
for i, (mot, freq) in enumerate(top_k, 1):
    print(f'{i:<6} {mot:<20} {freq:>6}')

Top 30 mots (RDD MapReduce) :
Rang   Mot                    Freq
-----------------------------------
1      appartement             691
2      maison                  662
3      situ                    543
4      tage                    373
5      calme                   326
6      cuisine                 304
7      salle                   284
8      meubl                   259
9      ces                     256
10     quartier                220
11     chambres                211
12     centre                  204
13     proximit                195
14     ville                   188
15     nov                     179
16     commerces               177
17     quip                    176
18     jour                    176
19     terrain                 175
20     lumineux                175
21     pied                    170
22     rement                  169
23     chambre                 169
24     louer                   164
25     rue                     164
26     enti             

In [9]:
# Top 15 mots par type de transaction
for transaction in ['vente', 'location']:
    print(f'\n--- Top 15 mots — {transaction.upper()} ---')
    df.filter(
        F.col('description').isNotNull() &
        (F.col('type_transaction') == transaction)
    ).select(
        F.explode(
            F.split(F.regexp_replace(F.lower('description'), r'[^a-z\s]', ' '), r'\s+')
        ).alias('mot')
    ).filter(F.length('mot') > 2) \
     .filter(~F.col('mot').isin(list(STOP_WORDS))) \
     .groupBy('mot').agg(F.count('*').alias('frequence')) \
     .orderBy(F.desc('frequence')) \
     .limit(15).show(truncate=False)


--- Top 15 mots — VENTE ---
+-----------+---------+
|mot        |frequence|
+-----------+---------+
|maison     |422      |
|appartement|350      |
|situ       |332      |
|tage       |205      |
|calme      |197      |
|terrain    |153      |
|ces        |150      |
|quartier   |129      |
|vendre     |116      |
|proximit   |108      |
|lumineux   |99       |
|chambres   |99       |
|commerces  |98       |
|couvrez    |97       |
|cuisine    |95       |
+-----------+---------+


--- Top 15 mots — LOCATION ---
+-----------+---------+
|mot        |frequence|
+-----------+---------+
|appartement|341      |
|maison     |240      |
|meubl      |238      |
|situ       |211      |
|cuisine    |209      |
|salle      |191      |
|tage       |168      |
|louer      |164      |
|quip       |136      |
|calme      |129      |
|ville      |119      |
|chambre    |116      |
|centre     |112      |
|chambres   |112      |
|eau        |108      |
+-----------+---------+



## 8. Export

In [10]:
out = os.path.join(DATA_DIR, 'immobilier_clean.csv')
if os.path.exists(out):
    shutil.rmtree(out) if os.path.isdir(out) else os.remove(out)
df.toPandas().to_csv(out, index=False, encoding='utf-8')
print(f'immobilier_clean.csv exporte : {df.count()} lignes')

immobilier_clean.csv exporte : 1790 lignes


In [11]:
spark.stop()
print('Spark termine.')

Spark termine.
